# 00 - File inventory

List all files in data/raw, peek at the first record of each, and check whether a user-bundle purchase file exists.

In [1]:
import os
import ast # parse JSON files - {"user_id": "abc"} to {"user_id": "abc"}
import pandas as pd

DATA_RAW = '../data/raw'
os.makedirs('../outputs/tables', exist_ok = True) 


In [2]:
file_records = []

for root, dirs, files in os.walk(DATA_RAW): #folders, sub-folders, files respectively
    for fname in files:
        fpath = os.path.join(root, fname)
        size_bytes = os.path.getsize(fpath)
        rel_path = os.path.relpath(fpath, DATA_RAW)
        file_records.append({
            'filename': fname,
              'rel_path': rel_path,
        'size_mb': round(size_bytes / (1024 ** 2), 3),
            'size_bytes': size_bytes
        })

inventory = pd.DataFrame(file_records).sort_values('size_mb', ascending=False)
print(inventory.to_string(index=False))


                    filename                          rel_path  size_mb  size_bytes
              steam_new.json steam_reviews.json\steam_new.json 4075.202  4273159526
                 archive.zip                       archive.zip 1396.571  1464410453
 australian_users_items.json       australian_users_items.json  527.472   553094816
australian_user_reviews.json      australian_user_reviews.json   25.100    26318774
            steam_games.json                  steam_games.json   20.261    21244748
            bundle_data.json                  bundle_data.json    0.796      834266


In [3]:
inventory.to_csv('../outputs/tables/file_inventory.csv') # saved as csv to tables/file_inventory.csv
print(f'{len(inventory)} files found')


6 files found


## Peek at the first record in each file

All files in this dataset use Python-style dicts (single quotes), so we parse with ast.literal_eval, not json.load.

In [4]:
SKIP_EXTENSIONS = {'.zip', '.gz', '.tar', '.bz2', '.rar', '.7z'} # source: stack overflow

def peek_pydict_file(fpath, n=1):
    if any(fpath.endswith(ext) for ext in SKIP_EXTENSIONS):
        return [{'skipped': 'binary file'}] # skip if compressed or binary
    records = []
    # try utf-8 first, fall back to latin-1 which never raises UnicodeDecodeError
    for enc in ('utf-8', 'latin-1'):
        try:
            with open(fpath, 'r', encoding=enc) as f:
                for i, line in enumerate(f):
                    if i >= n: # since n = 1, it stops after 1 line
                        break
                    line = line.strip()
                    if line:
                        try:
                            records.append(ast.literal_eval(line)) # parse
                        except Exception as e:
                            records.append({'parse_error': str(e), 'raw': line[:200]})
            break
        except UnicodeDecodeError:
            records = []
            continue
    return records # all parsed records

In [5]:
import json

for _, row in inventory.iterrows():
    fpath = os.path.join(DATA_RAW, row['rel_path'])
    if not os.path.isfile(fpath):
        continue
    print(f"\n{row['filename']} ({row['size_mb']} MB)")
    records = peek_pydict_file(fpath, n=1)
    if records and isinstance(records[0], dict): # if record exists and is a dict
        print('  keys:', list(records[0].keys()))
        print('  first record (truncated):')
        # print without nested lists, readability purposes
        sensitive = {'user_id', 'steam_id', 'user_url', 'username', 'text'}
        truncated = {
            k: ('[redacted]' if k in sensitive else v)
            for k, v in records[0].items() if not isinstance(v, list)
        }
        print(' ', json.dumps(truncated, indent=4, default=str))
    else:
        print('  ', records)



steam_new.json (4075.202 MB)
  keys: ['username', 'hours', 'products', 'product_id', 'page_order', 'date', 'text', 'early_access', 'page']
  first record (truncated):
  {
    "username": "[redacted]",
    "hours": 0.1,
    "products": 41,
    "product_id": "725280",
    "page_order": 0,
    "date": "2017-12-17",
    "text": "[redacted]",
    "early_access": false,
    "page": 1
}

archive.zip (1396.571 MB)
  keys: ['skipped']
  first record (truncated):
  {
    "skipped": "binary file"
}

australian_users_items.json (527.472 MB)
  keys: ['user_id', 'items_count', 'steam_id', 'user_url', 'items']
  first record (truncated):
  {
    "user_id": "[redacted]",
    "items_count": 277,
    "steam_id": "[redacted]",
    "user_url": "[redacted]"
}

australian_user_reviews.json (25.1 MB)
  keys: ['user_id', 'user_url', 'reviews']
  first record (truncated):
  {
    "user_id": "[redacted]",
    "user_url": "[redacted]"
}

steam_games.json (20.261 MB)
  keys: ['publisher', 'genres', 'app_name', '

## Check for a user-bundle purchase file

The associated paper reports 87,565 bundle purchases by 29,634 users. If those records exist as a file, it would have both a user id and a bundle id.

In [6]:
print('files whose names suggest user-bundle purchase content:')
for _, row in inventory.iterrows():
    fl = row['filename'].lower()
    if 'bundle' in fl and 'user' in fl:
        print(' ', row['filename'], '-', row['size_mb'], 'MB')

print('\nscanned all files for records containing both user and bundle fields')
for _, row in inventory.iterrows():
    fpath = os.path.join(DATA_RAW, row['rel_path'])
    if not os.path.isfile(fpath):
        continue
    records = peek_pydict_file(fpath, n=1)
    if not records or not isinstance(records[0], dict):
        continue
    keys_lower = [k.lower() for k in records[0].keys()]
    has_user = any(kw in k for k in keys_lower for kw in ['user', 'steamid'])
    has_bundle = any('bundle' in k for k in keys_lower)
    if has_user and has_bundle:
        print(f' user+bundle: {row["filename"]} -- keys: {list(records[0].keys())}')
    elif has_bundle:
        print(f'  bundle only: {row["filename"]} -- keys: {list(records[0].keys())}')
    elif has_user:
        print(f'  user only:   {row["filename"]} -- keys: {list(records[0].keys())}')


files whose names suggest user-bundle purchase content:

scanned all files for records containing both user and bundle fields
  user only:   steam_new.json -- keys: ['username', 'hours', 'products', 'product_id', 'page_order', 'date', 'text', 'early_access', 'page']
  user only:   australian_users_items.json -- keys: ['user_id', 'items_count', 'steam_id', 'user_url', 'items']
  user only:   australian_user_reviews.json -- keys: ['user_id', 'user_url', 'reviews']
  bundle only: bundle_data.json -- keys: ['bundle_final_price', 'bundle_url', 'bundle_price', 'bundle_name', 'bundle_id', 'items', 'bundle_discount']


## Findings

- Files: bundle_data.json, australian_users_items.json, australian_user_reviews.json, steam_games.json
- bundle_data keys: bundle_id, bundle_name, bundle_price, bundle_final_price, bundle_discount, items
- User-bundle purchase file: does not exist, so we build demand proxies in notebook 03
- The item price field in bundle_data is `discounted_price` (the standalone game price, not a bundle-specific price)